# 13 — Multivariate forcing: comparison and ablations

Compares the SST-only FNO against multivariate arms under **identical target
dates and a matched tuning budget**.

The acceptance question is deliberately strict: does exogenous forcing add
*out-of-sample information beyond SST persistence*? A multivariate model that
merely fits better, or that beats the SST-only model while both lose to
persistence, has not answered it.

Prerequisite: notebook `12` for the audit and the OISST auxiliary download.

In [ ]:
from pathlib import Path
import json

import numpy as np
import torch
from torch.utils.data import DataLoader

from oisst_fno.data import ForecastSpec, Standardizer, open_oisst, temporal_split
from oisst_fno.experiment import set_global_seed
from oisst_fno.metrics import daily_rmse, moving_block_bootstrap_mean_ci, rmse, skill_score
from oisst_fno.multivariate import (
    MultivariateWindowDataset,
    PerVariableStandardizer,
    channel_layout,
    standard_ablations,
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
set_global_seed(SEED)

TRAIN_END = "2024-12-31"
VALIDATION_END = "2025-12-31"
SPEC = ForecastSpec(lookback_days=14, horizon_days=7)

for arm in standard_ablations():
    print(f"{arm.name:>22}  {list(arm.exogenous)}")
    print(f"{'':>22}  {arm.rationale}")

## Which arms can run

The OISST auxiliary channels need no credentials. The ERA5 arms require a CDS
personal access token, so they are skipped automatically when the file is
absent — rather than silently substituting something else.

In [ ]:
# Which arms can actually run depends on what has been downloaded.
AUX_CANDIDATES = sorted(RAW.glob("oisst_aux_*_ne_atlantic.nc"))
ERA5_CANDIDATES = sorted(RAW.glob("era5_*_ne_atlantic.nc"))

print("OISST auxiliary file:", AUX_CANDIDATES[-1].name if AUX_CANDIDATES else "MISSING (run notebook 12)")
print("ERA5 file           :", ERA5_CANDIDATES[-1].name if ERA5_CANDIDATES else "MISSING (needs CDS credentials)")

if not AUX_CANDIDATES:
    raise FileNotFoundError("Run notebook 12 first to download the OISST auxiliary fields.")

aux = open_oisst(AUX_CANDIDATES[-1])
sst = aux["sst"]

# Runnable arms without ERA5: SST-only, and SST plus the OISST confidence channel.
RUNNABLE = [("sst-only", ()), ("sst+err", ("err",))]
if ERA5_CANDIDATES:
    RUNNABLE += [(arm.name, arm.exogenous) for arm in standard_ablations() if arm.exogenous]
print("\narms to run:", [name for name, _ in RUNNABLE])

## Building the arms

SST normalisation is fitted on the training split only, as in notebook `07`.
Exogenous channels get **per-variable** statistics, also training-only, because
their units differ by orders of magnitude — pressure in pascals beside wind in
m/s — and one global statistic would let a single channel dominate the input
scale.

In [ ]:
def build_arm(exogenous_names):
    """Build train/validation/test datasets for one arm.

    Every arm shares the same split boundaries, the same window specification, and
    therefore the same target dates. Only the exogenous channels differ.
    """
    train_sst, val_sst, test_sst = temporal_split(sst, TRAIN_END, VALIDATION_END)

    # SST normalisation is fitted on the training split only, exactly as in notebook 07.
    sst_scaler = Standardizer.fit(train_sst.values)

    def stack(split_sst, exo_scaler=None):
        if not exogenous_names:
            return sst_scaler.transform(split_sst.values), None, exo_scaler
        selected = aux[list(exogenous_names)].sel(time=split_sst["time"])
        if exo_scaler is None:
            exo_scaler = PerVariableStandardizer.fit(selected, tuple(exogenous_names))
        standardized = exo_scaler.transform(selected)
        forcing = np.stack(
            [standardized[name].values for name in exogenous_names], axis=1
        ).astype(np.float32)
        return sst_scaler.transform(split_sst.values), forcing, exo_scaler

    train_values, train_forcing, exo_scaler = stack(train_sst)  # fits on train only
    val_values, val_forcing, _ = stack(val_sst, exo_scaler)
    test_values, test_forcing, _ = stack(test_sst, exo_scaler)

    return {
        "train": MultivariateWindowDataset(train_values, SPEC, train_forcing, tuple(exogenous_names)),
        "val": MultivariateWindowDataset(val_values, SPEC, val_forcing, tuple(exogenous_names)),
        "test": MultivariateWindowDataset(test_values, SPEC, test_forcing, tuple(exogenous_names)),
        "sst_scaler": sst_scaler,
        "exo_scaler": exo_scaler,
        "channels": channel_layout(SPEC.lookback_days, tuple(exogenous_names)),
    }


arms = {name: build_arm(exo) for name, exo in RUNNABLE}
for name, arm in arms.items():
    print(f"{name:>10}: {len(arm['train'])} train / {len(arm['val'])} val / {len(arm['test'])} test windows")
    print(f"{'':>10}  channels: {arm['channels']}")

### Identical targets

Asserted, not assumed. If the arms predicted different dates the comparison
would be meaningless, and a silent off-by-one in window construction is exactly
the kind of bug that produces a flattering result.

In [ ]:
# Identical target dates across arms is the whole point of the comparison; assert it.
reference = arms["sst-only"]
for name, arm in arms.items():
    assert len(arm["test"]) == len(reference["test"]), f"{name} has a different test length"
    x_ref, y_ref, m_ref = reference["test"][0]
    x_arm, y_arm, m_arm = arm["test"][0]
    assert torch.equal(y_ref, y_arm), f"{name} predicts a different target"
    assert torch.equal(m_ref, m_arm), f"{name} uses a different mask"
print("all arms share identical targets and masks on the test split")

### Matched budget

Same architecture, optimizer, schedule, and epoch count for every arm. Adding
channels does add parameters in the lifting layer; that difference is printed
rather than hidden, since "the bigger model won" is the alternative explanation
this comparison has to rule out.

In [ ]:
# Matched tuning budget: every arm gets the same architecture, optimizer settings, and
# epoch count. The only difference is input channel count, which changes the first
# layer's parameters slightly - report that, do not hide it.
from oisst_fno.metrics import parameter_count
from oisst_fno.model import FNO2d

BASE_MODEL = {"out_channels": 1, "width": 48, "modes_y": 16, "modes_x": 16, "depth": 4, "padding": 8}

print(f"{'arm':>10} {'in_channels':>12} {'parameters':>14}")
for name, arm in arms.items():
    in_channels = len(arm["channels"])  # history + exogenous + mask
    model = FNO2d(in_channels=in_channels, **BASE_MODEL)
    print(f"{name:>10} {in_channels:>12} {parameter_count(model):>14,}")

## Training and evaluation

Each arm is a full training run, so this is the expensive part. **No numbers are
reported until it has actually been executed** — the cell below raises rather
than emitting placeholder results.

In [ ]:
# Training every arm is the expensive step. Reuse notebook 07's loop verbatim per arm,
# with the same seed, epochs, patience, and optimizer settings, then evaluate all arms
# on the identical test targets.
#
# Left as an explicit, un-run cell because each arm is a full training run and the
# result must not be fabricated. Fill in and execute when running the experiment.
raise NotImplementedError(
    "Train each arm with notebook 07's loop under a matched budget, then continue. "
    "No results are reported until this has actually been run."
)

In [ ]:
# Once predictions exist, the comparison is paired on identical target dates and the
# uncertainty respects serial dependence. Persistence remains the primary null model.
#
#   skill_vs_persistence = skill_score(rmse(model), rmse(persistence))
#   paired_difference    = daily_rmse(persistence, truth, mask) - daily_rmse(model, truth, mask)
#   interval             = moving_block_bootstrap_mean_ci(paired_difference, block_size=14)
#
# The question is NOT whether the multivariate arm beats the SST-only arm on training
# loss. It is whether it adds out-of-sample information beyond SST persistence, with an
# interval that excludes zero.
print("Evaluation recipe defined above; run after training.")

## Analysis to run afterwards

1. **Aggregate skill versus persistence**, per arm, on the untouched test split.
2. **Paired daily RMSE differences** with a moving-block bootstrap interval, so
   serial dependence is respected.
3. **Seasonal breakdown** — are gains concentrated in particular seasons?
4. **High-change periods** — days where persistence performs worst, with the
   threshold estimated on training and validation only, never on test.

## Reporting rules

- A gain concentrated in one season, or one regime, is a **finding about that
  regime**, not a general claim.
- Subgroup splits chosen after seeing test results are exploratory and must be
  labelled as such.
- **Predictive is not causal.** Improvement from a wind channel does not attribute
  SST change to wind forcing; ERA5 and OISST are both analysis products and may
  share assimilated observations.
- A negative result — exogenous forcing adds nothing beyond SST history — is a
  valid and reportable outcome.